# Milestone M2 - Audit Data Menyeluruh, EDA, Uji Kebocoran & Model Baseline (Topik T3-A)
**Kelompok 6 - Deep Learning A (Sistem Informasi, Universitas Trunojoyo Madura)**  
*Anggota:* Attala Alif Ramadhani Tri Hida (`230441100144`), Nafaul Hernanda Romadlona (`240441100125`), M. Rafly Kurniawan (`240441100086`), Naufal Husain (`240441100038`)

Notebook ini dijalankan bersih dari awal untuk:
1. Menghubungkan Google Colab ke folder Google Drive target (`101PbkgjCD-y_OaZPzRcqZzdgjoCf2ld8`).
2. Mengunduh dataset resmi Zheng et al. (2018) (Dataset 1 & Dataset 2) dan melakukan **audit data menyeluruh** (seluruh dimensi unik, masker mentah/stray pixel, sebaran luas & jumlah objek, rasio N:C, pemeriksaan kebocoran SHA-256 + dHash, dan tabel per citra 400 baris).
3. Menampilkan 5 grafik EDA termasuk verifikasi visual konversi masker pada **24 citra** (min. 20 citra sesuai Kisi-Kisi 3.1).
4. Melatih & mengevaluasi **1 model baseline** (`U-Net ResNet-34`, `D1 -> D2`), menyimpan kurva latih, bobot `.pth`, evaluasi per citra, dan menyinkronkan seluruh hasilnya langsung ke Google Drive.

In [ ]:
import os
import sys
import subprocess
import pandas as pd

REPO_URL = "https://github.com/attaramadhani/UAS-DL-Kelompok6-T3A-WBC-DomainShift.git"
REPO_NAME = "UAS-DL-Kelompok6-T3A-WBC-DomainShift"
GDRIVE_FOLDER_ID = "101PbkgjCD-y_OaZPzRcqZzdgjoCf2ld8"
RUN_FROM_SCRATCH = True

if "google.colab" in sys.modules:
    if os.path.basename(os.getcwd()) != REPO_NAME:
        if not os.path.exists(REPO_NAME):
            subprocess.run(["git", "clone", REPO_URL], check=True)
        os.chdir(REPO_NAME)
    subprocess.run(["git", "pull", "--ff-only"], check=False)

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", 2000)
pd.set_option("display.max_colwidth", 120)

import torch
from src.gdrive_sync import GDriveFolderSync

print(
    "PyTorch:",
    torch.__version__,
    "| CUDA:",
    torch.cuda.is_available(),
    torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
)

gdrive = GDriveFolderSync(folder_id=GDRIVE_FOLDER_ID, milestone_subdir="M2_Audit_EDA_Baseline")
gdrive.connect(require_in_colab=True)

## 1. Unduh Dataset Resmi (Mendeley) & Tampilkan Keseluruhan Audit Dataset (`seed=42`)
Seluruh tabel di bawah menampilkan hasil audit lengkap tanpa pemotongan kolom/baris.

In [ ]:
from IPython.display import display
from src.audit_eda import main as run_audit

audit_out = run_audit()
tables = audit_out["tables"]
df_audit = audit_out["per_image_df"]

print("=== 1. RINGKASAN EKSEKUTIF DOMAIN (D1 vs D2) ===")
display(tables["domain_overview"])

print("\n=== 2. KOMPOSISI SPLIT TETAP (seed=42, GROUP-AWARE) ===")
display(tables["split_composition"])

print("\n=== 3. LAPORAN PEMERIKSAAN KEBOCORAN DATA & DUPLIKAT (SHA-256 & dHash) ===")
display(tables["leakage_report"])

print("\n=== 4. SELURUH DIMENSI UNIK CITRA D1 & D2 (LENGKAP TANPA TERPOTONG) ===")
display(tables["dimensions_complete"])

print("\n=== 5. STATISTIK KELAS, JUMLAH KOMPONEN, LUAS OBJEK & RASIO N:C ===")
display(tables["object_statistics"])

print("\n=== 6. DAFTAR BERKAS MASKER DENGAN PIKSEL STRAY (<64) SEBELUM KONVERSI ===")
display(tables["stray_mask_files"])

print("\n=== 7. KESEIMBANGAN DISTRIBUSI ANTAR-SUBSET (TRAIN / VAL / TEST) ===")
split_balance = (
    df_audit.groupby(["domain", "split"], as_index=False)
    .agg(
        n_images=("id", "count"),
        bg_pct_mean=("bg_pct", "mean"),
        cyto_pct_mean=("cyto_pct", "mean"),
        nuc_pct_mean=("nuc_pct", "mean"),
        nc_ratio_mean=("nc_ratio", "mean"),
        fg_area_px_mean=("fg_area_px", "mean"),
    )
    .round(2)
)
display(split_balance)

print(f"\n=== 8. TABEL AUDIT LENGKAP PER CITRA ({len(df_audit)} CITRA D1 + D2) ===")
audit_view_cols = [
    "domain", "id", "split", "width", "height", "bg_pct", "cyto_pct", "nuc_pct",
    "nuc_area_px", "cyto_area_px", "nc_ratio", "nuc_components", "has_stray_pixels", "dhash", "sha256",
]
display(df_audit[audit_view_cols])

if gdrive.connected:
    n_up = gdrive.sync_paths([
        "logs/data_audit_report.json",
        "logs/data_audit_per_image.csv",
        "splits",
        "docs/figures",
    ])
    print(f"[GDrive] {n_up} berkas audit, split CSV, & grafik EDA M2 langsung tersimpan ke Google Drive.")

## 2. Grafik EDA, Verifikasi Visual Konversi 24 Citra & Preview Macenko / Reinhard

In [ ]:
from IPython.display import Image, display

for fig_name in [
    "fig1_sample_overlay_d1_d2.png",
    "fig1b_mask_conversion_audit_24.png",
    "fig2_eda_distributions.png",
    "fig2b_object_size_and_nc_distributions.png",
    "fig3_stain_norm_preview.png",
]:
    print("===", fig_name, "===")
    display(Image(filename=f"docs/figures/{fig_name}"))

## 3. Latih & Evaluasi 1 Model Baseline (`U-Net ResNet-34`: Train `D1` -> Uji In-Domain `D1` & Cross-Domain `D2`)

In [ ]:
import os
import pandas as pd
from IPython.display import Image, display
from src.dataset import download_and_extract_zheng
from src.trainer import run_experiment
from src.xai_and_errors import (
    records_to_dataframe,
    save_baseline_preview_grid,
    save_learning_curve_plot,
    summarize_by_object_size,
    summarize_error_categories,
)

M2_LOG_PATH = "logs/m2_baseline_runs.csv"
if RUN_FROM_SCRATCH and os.path.exists(M2_LOG_PATH):
    os.remove(M2_LOG_PATH)

domain_dirs = download_and_extract_zheng("data")
split_csvs = {"D1": "splits/zheng_d1_split.csv", "D2": "splits/zheng_d2_split.csv"}

baseline_res = run_experiment(
    run_name="M2_Baseline_UNet_D1_to_D2",
    owner="Kelompok 6 (Baseline M2)",
    strategy="NoAdapt",
    domain_dirs=domain_dirs,
    split_csvs=split_csvs,
    train_domain="D1",
    target_domain="D2",
    arch="unet",
    encoder_name="resnet34",
    loss_name="ce_dice",
    aug_mode="geom",
    seed=42,
    epochs=30,
    log_csv_path=M2_LOG_PATH,
    save_weights_path="weights/M2_Baseline_UNet_D1_to_D2.pth",
    capture_test_records=True,
    history_dir="logs/histories",
)

df_m2 = pd.read_csv(M2_LOG_PATH)
df_m2["delta_miou_drop"] = (df_m2["in_miou_fg"] - df_m2["cross_miou_fg"]).round(4)
print("=== RINGKASAN METRIK LENGKAP M2 BASELINE ===")
display(df_m2.T.rename(columns={0: "Nilai_M2_Baseline"}))

df_in_rec = records_to_dataframe(baseline_res["in_records"], strategy="Baseline_M2", domain_label="D1_InDomain")
df_cr_rec = records_to_dataframe(baseline_res["cross_records"], strategy="Baseline_M2", domain_label="D2_CrossDomain")
df_per_img = pd.concat([df_in_rec, df_cr_rec], ignore_index=True)
df_per_img.to_csv("logs/m2_baseline_per_image.csv", index=False)

print("\n=== KATEGORI KESALAHAN BASELINE M2 (D2 TEST) ===")
print(summarize_error_categories(baseline_res["cross_records"]))

print("\n=== PERFORMA BASELINE BERDASARKAN UKURAN SEL (D2 TEST) ===")
size_df_m2 = summarize_by_object_size(baseline_res["cross_records"], strategy="Baseline_M2", domain_label="D2_CrossDomain")
size_df_m2.to_csv("logs/m2_baseline_size_breakdown.csv", index=False)
display(size_df_m2)

save_learning_curve_plot(
    {"M2 Baseline (U-Net D1->D2)": "logs/histories/M2_Baseline_UNet_D1_to_D2.csv"},
    out_path="docs/figures/fig_m2_baseline_learning_curve.png",
    title="Kurva Pelatihan & Validasi M2 Baseline (U-Net ResNet-34, 30 Epoch)",
)
save_baseline_preview_grid(
    baseline_res["in_records"],
    baseline_res["cross_records"],
    out_path="docs/figures/fig_m2_baseline_predictions.png",
    n_each=2,
    first_label="In-Domain (D1)",
    second_label="Cross-Domain (D2)",
)
display(Image(filename="docs/figures/fig_m2_baseline_learning_curve.png"))
display(Image(filename="docs/figures/fig_m2_baseline_predictions.png"))

## 4. Bundel Arsip & Simpan Keseluruhan Hasil M2 ke Google Drive Target

In [ ]:
import os
import zipfile
import pandas as pd
from IPython.display import display

bundle_path = "m2_results_bundle.zip"
with zipfile.ZipFile(bundle_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in ("logs", "splits", "docs/figures", "configs"):
        if not os.path.exists(folder):
            continue
        for root, _, files in os.walk(folder):
            for fname in sorted(files):
                zf.write(os.path.join(root, fname))

print("Arsip M2 dibuat di:", os.path.abspath(bundle_path))
if gdrive.connected:
    uploaded = gdrive.sync_paths(["logs", "splits", "docs/figures", "weights", "configs", bundle_path])
    print(f"[OK] {uploaded} berkas M2 tersimpan ke Google Drive: https://drive.google.com/drive/folders/{GDRIVE_FOLDER_ID}")
    remote_df = pd.DataFrame(gdrive.list_remote_files())
    if not remote_df.empty:
        print(f"Total ukuran di '{gdrive.milestone_subdir}': {remote_df['size_mb'].sum():.2f} MB ({len(remote_df)} berkas)")
        display(remote_df[["drive_path", "size_mb", "file_id"]])